<div align="center">
    <img src="../../../media/a365-agents.png" width="100%" alt="Microsoft Foundry workshop / lab / sample">
</div>

# Build a pro-code Foundry agent that is fully in sync with Agent 365

In this lab you will:

1. **Register an Entra Agent ID** &mdash; a workload identity for your AI agent.
2. **Build a pro-code Foundry agent** with the `azure-ai-projects` SDK and two function tools.
3. **Bind** the Foundry agent to the Entra Agent ID so its runs execute under the agent identity.
4. **Author the Agent 365 manifest** in code (no hidden YAML).
5. **Publish to Agent 365** with the Microsoft 365 Agents Toolkit (`atk`).
6. **Verify the agent is fully in sync** across Entra, Foundry, and Agent 365.
7. **Apply and test five Agent 365 policies**: DLP, Conditional Access, tool allow-list, audit, lifecycle.
8. **Clean up** so re-running the notebook is idempotent.

## Architecture

```
        ┌──────────────────────┐        ┌────────────────────────┐
        │   Entra ID           │        │   Microsoft Foundry    │
        │  (Agent ID = appId)  │◀──bind─│  (Agent runtime)       │
        └──────────┬───────────┘        └────────────┬───────────┘
                   │                                 │
                   │ identity                        │ runtime
                   ▼                                 ▼
        ┌──────────────────────────────────────────────────────────┐
        │                Agent 365 (governance plane)              │
        │  manifest · DLP · Conditional Access · audit · lifecycle │
        └──────────────────────────────────────────────────────────┘
```

**Fully in sync** means the same logical agent is observable in all three planes
with consistent identifiers (`entraAgentId == manifest.identity.entraAgentId`,
`foundryAgentId == manifest.runtime.agentId`).

> **Run the main workshop first.** This lab assumes you completed
> [`src/workshop/README.md`](../../workshop/README.md) and have a populated
> [`src/workshop/.env`](../../workshop/.env).

## Prerequisites

Before you start this lab, make sure the following prerequisites are in place.

| # | Requirement | How to verify / get it |
| --- | --- | --- |
| 1 | Completed the main workshop and populated the `.env` file | See [`src/workshop/README.md`](../../workshop/README.md) |
| 2 | Microsoft 365 tenant with **Agent 365** enabled | Microsoft 365 admin center → **Agents** |
| 3 | Signed-in user has the **AI Administrator** or **Global Administrator** role | Entra admin center → **Roles & administrators** |
| 4 | Signed-in user can manage owned app registrations | Verify with `az ad signed-in-user show` |
| 5 | `node` version 20 or higher and `npm` installed | Run `node --version` |
| 6 | Microsoft 365 Agents Toolkit CLI (`atk`) installed | Installed in **Step 5** of this lab |
| 7 | Azure CLI login completed in this terminal | Run `az account show` |
| 8 | Microsoft Cognitive Services resource provider registered for the subscription | See command below |

Register the resource provider once per Azure subscription:

```bash
az provider register --namespace Microsoft.CognitiveServices

## Select the right Python kernel before you run anything

> [!IMPORTANT]
> Before executing any cell, make sure the notebook is using the **workshop's
> shared virtual environment** — not the system Python and not a fresh
> auto-created kernel. All other labs in this repo use the same `.venv` so
> packages installed once are reused everywhere.

1. Click the kernel picker in the top-right of this notebook (it may say
   *Select Kernel* or show a different interpreter).
2. Choose **Python Environments…** → **`.venv (Python 3.x)`** located at
   the repo root: `/workspaces/Microsoft-Foundry/.venv/bin/python`.
3. If you don't see it, run `source .venv/bin/activate` in a terminal once,
   then click *Select Another Kernel…* → *Python Environments…* and pick it.

### First-time package install takes 1–2 minutes

If this is the **first notebook you run** in your Codespace / devcontainer,
the next cell (`%pip install -r requirements.txt`) will pull down
`azure-ai-projects`, `azure-identity`, `msgraph-sdk`, `httpx`, and their
transitive dependencies. Expect **1–2 minutes** the first time. Subsequent
runs are nearly instant because the packages are cached in the shared
`.venv`. Wait for the cell to finish (the `[*]` indicator turns into a
number) before moving on.

## Setup &mdash; install lab dependencies

Re-uses the workshop-wide `.venv` at the repo root. Run the next cell once.

In [40]:
%pip install -q -r requirements.txt


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [41]:
import json
import os
import sys
import time
import subprocess
from pathlib import Path

import httpx
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

# Load the workshop .env (single source of truth for the Foundry project).
WORKSHOP_ENV = Path("../../workshop/.env").resolve()
assert WORKSHOP_ENV.exists(), f"Run the main workshop first — {WORKSHOP_ENV} not found."
load_dotenv(WORKSHOP_ENV)

PROJECT_ENDPOINT = os.environ["PROJECT_ENDPOINT"]
MODEL_DEPLOYMENT_NAME = os.environ["AGENT_MODEL_DEPLOYMENT_NAME"]
AZURE_SUBSCRIPTION_ID = os.environ["AZURE_SUBSCRIPTION_ID"]
AZURE_RESOURCE_GROUP_NAME = os.environ["AZURE_RESOURCE_GROUP_NAME"]

# Make agent_app.py importable.
sys.path.insert(0, str(Path.cwd()))
import agent_app  # noqa: E402

credential = DefaultAzureCredential()
print("Foundry endpoint:", PROJECT_ENDPOINT)
print("Model deployment:", MODEL_DEPLOYMENT_NAME)

Foundry endpoint: https://aif-aiagents-vbds.services.ai.azure.com/api/projects/workshop-project
Model deployment: gpt4o


## Step 1 &mdash; Register an Entra Agent ID

An **Entra Agent ID** is a first-class workload identity for an AI agent. It is
what Conditional Access targets, what audit logs attribute actions to, and what
Agent 365 binds the manifest to. It is **not** a regular service principal &mdash;
the Microsoft Graph `applications` resource is created with the
`agentApplication` workload type.

We use **detect-and-reuse** so re-running the notebook does not create duplicates.

In [42]:
ENTRA_AGENT_DISPLAY_NAME = "foundry-lab-agent"
GRAPH_ROOT = "https://graph.microsoft.com"

def graph_token() -> str:
    return credential.get_token("https://graph.microsoft.com/.default").token

def graph_request(
    method: str,
    path: str,
    *,
    api_version: str = "v1.0",
    params: dict | None = None,
    headers: dict | None = None,
    **kwargs
) -> httpx.Response:
    """
    Helper voor Microsoft Graph API calls.
    - path mag een volledige URL zijn, of starten met /v1.0/ of /beta/, of een relatieve path.
    - api_version: 'v1.0' (default) of 'beta'.
    - params: optionele dict met query parameters.
    - headers: extra headers (optioneel).
    """
    token = graph_token()
    if headers is None:
        headers = {}
    headers = dict(headers)  # copy
    headers["Authorization"] = f"Bearer {token}"
    headers.setdefault("Accept", "application/json")

    # URL logica
    if path.startswith("https://"):
        url = path
    elif path.startswith("/v1.0/") or path.startswith("/beta/"):
        url = f"{GRAPH_ROOT}{path}"
    else:
        if not path.startswith("/"):
            path = "/" + path
        url = f"{GRAPH_ROOT}/{api_version}{path}"

    response = httpx.request(
        method,
        url,
        headers=headers,
        params=params,
        timeout=30,
        **kwargs
    )

    if response.status_code >= 400:
        print("Graph request failed")
        print("Status:", response.status_code)
        print("URL:", response.request.url)
        print("Body:", response.text)

    return response

Grant the Agent ID the **Cognitive Services User** role on the Foundry project
so it can execute runs under its own identity. This is the same role pattern as
`DefaultAzureCredential` for human users &mdash; only the principal changes.

In [43]:
scope = (
    f"/subscriptions/{AZURE_SUBSCRIPTION_ID}"
    f"/resourceGroups/{AZURE_RESOURCE_GROUP_NAME}"
)
result = subprocess.run(
    [
        "az", "role", "assignment", "create",
        "--assignee", ENTRA_AGENT_ID,
        "--role", "Cognitive Services User",
        "--scope", scope,
    ],
    capture_output=True, text=True,
)
# Idempotent: ignore "already exists".
if result.returncode != 0 and "already exists" not in result.stderr.lower():
    raise RuntimeError(result.stderr)
print("Role assignment OK for", ENTRA_AGENT_ID)

Role assignment OK for eab91a3d-3034-48d4-b540-65c56be4b9bd


## Step 2 &mdash; Build the pro-code Foundry agent

The agent definition lives in [`agent_app.py`](agent_app.py) so the notebook
stays focused on flow. Two function tools are registered:

* `lookup_policy` &mdash; deterministic HR/IT policy lookup (used in the **tool allow-list** policy test).
* `summarize_ticket` &mdash; summarises a fake support ticket (used in the **DLP** policy test).

Open `agent_app.py` to see exactly what code is shipped to Foundry.

In [44]:
import importlib
importlib.reload(agent_app)  # picks up edits to agent_app.py without restarting the kernel

project_client = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=credential,
)
# The Responses API is exposed via the OpenAI-compatible client minted
# from the project client. Foundry agents are addressed via `extra_body`
# (`agent_reference` -> name + version), not via `agent_id`.
openai_client = project_client.get_openai_client()

handle = agent_app.build_or_get_agent(project_client, MODEL_DEPLOYMENT_NAME)
FOUNDRY_AGENT_NAME = handle.name
FOUNDRY_AGENT_VERSION = handle.version
print("Foundry agent:", handle)

Foundry agent: AgentHandle(name='foundry-lab-agent', version='2', model='gpt4o', created=False)


Quick smoke test &mdash; run a single turn to confirm the agent + tools work
**before** we add governance.

In [45]:
response = agent_app.run_turn(
    openai_client,
    MODEL_DEPLOYMENT_NAME,
    handle,
    "What is our remote-work policy?",
)
print(agent_app.final_text(response))

Our remote work policy allows employees to work remotely up to 3 days per week, provided they have manager approval.


## Step 3 &mdash; Bind the Foundry agent to the Entra Agent ID

By default a Foundry agent runs under the *developer's* identity. To make it
governable by Agent 365 we record the **Entra Agent ID** on the Foundry agent
so the runtime can issue tokens under the agent's workload identity.

The binding is stored on the agent's `instance_identity` (read-only, set at
agent creation time) and mirrored as metadata so we can inspect it from any
SDK call. This is what makes Conditional Access, audit attribution, and DLP
policies actually apply to the agent &mdash; they all key off the Entra Agent ID.

In [46]:
from azure.ai.projects.models import PromptAgentDefinition

# Re-publish the agent with metadata that records the Entra Agent ID. The
# Foundry control plane uses this metadata to provision the runtime's
# workload identity. `create_version` is idempotent for our purposes — each
# call produces a new immutable version; the latest version is what runs.
binding_metadata = {
    "entraAgentId": ENTRA_AGENT_ID,
    "identityType": "EntraAgentId",
}

bound_version = project_client.agents.create_version(
    agent_name=FOUNDRY_AGENT_NAME,
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT_NAME,
        instructions=agent_app.AGENT_INSTRUCTIONS,
        tools=[agent_app.lookup_policy_tool, agent_app.summarize_ticket_tool],
    ),
    description=agent_app.AGENT_DESCRIPTION,
    metadata=binding_metadata,
)
FOUNDRY_AGENT_VERSION = str(bound_version.version)
handle = agent_app.AgentHandle(
    name=FOUNDRY_AGENT_NAME,
    version=FOUNDRY_AGENT_VERSION,
    model=MODEL_DEPLOYMENT_NAME,
    created=False,
)

# Confirm the binding round-trips on the AgentDetails record.
bound = project_client.agents.get(agent_name=FOUNDRY_AGENT_NAME)
print(json.dumps({
    "agent_name": bound.name,
    "latest_version": FOUNDRY_AGENT_VERSION,
    "instance_identity": bound.instance_identity,
    "binding_metadata": binding_metadata,
}, indent=2, default=str))

{
  "agent_name": "foundry-lab-agent",
  "latest_version": "2",
  "instance_identity": "{'principal_id': '890ff42b-bd1f-472f-ac31-6a21875cd846', 'client_id': '890ff42b-bd1f-472f-ac31-6a21875cd846'}",
  "binding_metadata": {
    "entraAgentId": "eab91a3d-3034-48d4-b540-65c56be4b9bd",
    "identityType": "EntraAgentId"
  }
}


## Step 4 &mdash; Author the Agent 365 manifest

The Agent 365 manifest tells the governance plane:

* who the agent is (**identity** &rarr; Entra Agent ID),
* where it runs (**runtime** &rarr; Foundry endpoint + agent id),
* what actions it can take (so admins can build allow-lists).

We generate the manifest from code so participants see every required field.

In [47]:
MANIFEST_DIR = Path("manifest")
MANIFEST_DIR.mkdir(exist_ok=True)

manifest = {
    "$schema": "https://developer.microsoft.com/json-schemas/agent365/v1.0/manifest.json",
    "manifestVersion": "1.0",
    "id": "foundry-lab-agent",
    "version": "1.0.0",
    "developer": {
        "name": "Contoso Agent Lab",
        "websiteUrl": "https://contoso.example.com",
        "privacyUrl": "https://contoso.example.com/privacy",
        "termsOfUseUrl": "https://contoso.example.com/terms",
    },
    "name": {"short": "Foundry Lab", "full": "Foundry Lab Agent (pro-code)"},
    "description": {
        "short": "Pro-code Foundry agent governed by Agent 365.",
        "full": agent_app.AGENT_DESCRIPTION,
    },
    "identity": {"entraAgentId": ENTRA_AGENT_ID},
    "runtime": {
        "type": "foundry",
        "endpoint": PROJECT_ENDPOINT,
        "agentName": FOUNDRY_AGENT_NAME,
        "agentVersion": FOUNDRY_AGENT_VERSION,
        "model": MODEL_DEPLOYMENT_NAME,
    },
    "actions": [
        {"id": "lookup_policy", "description": "Read company policy snippets."},
        {"id": "summarize_ticket", "description": "Summarise a support ticket."},
    ],
    "permissions": {
        "resourceSpecific": [],
        "orgWide": [],
    },
    "validDomains": ["contoso.example.com"],
}
(MANIFEST_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))

declarative_agent = {
    "$schema": "https://developer.microsoft.com/json-schemas/agent365/v1.0/declarativeAgent.json",
    "name": "Foundry Lab Agent",
    "description": agent_app.AGENT_DESCRIPTION,
    "instructions": agent_app.AGENT_INSTRUCTIONS,
    "capabilities": [
        {
            "name": "FoundryRuntime",
            "agentName": FOUNDRY_AGENT_NAME,
            "agentVersion": FOUNDRY_AGENT_VERSION,
        },
    ],
    "actions": [a["id"] for a in manifest["actions"]],
}
(MANIFEST_DIR / "declarativeAgent.json").write_text(json.dumps(declarative_agent, indent=2))

print("Manifest written to", MANIFEST_DIR.resolve())
print(json.dumps(manifest, indent=2))

Manifest written to /workspaces/Microsoft-Foundry/src/samples/create-agent365-managed-agents/manifest
{
  "$schema": "https://developer.microsoft.com/json-schemas/agent365/v1.0/manifest.json",
  "manifestVersion": "1.0",
  "id": "foundry-lab-agent",
  "version": "1.0.0",
  "developer": {
    "name": "Contoso Agent Lab",
    "websiteUrl": "https://contoso.example.com",
    "privacyUrl": "https://contoso.example.com/privacy",
    "termsOfUseUrl": "https://contoso.example.com/terms"
  },
  "name": {
    "short": "Foundry Lab",
    "full": "Foundry Lab Agent (pro-code)"
  },
  "description": {
    "short": "Pro-code Foundry agent governed by Agent 365.",
    "full": "Pro-code Foundry lab agent registered with an Entra Agent ID and published to Agent 365 for governance."
  },
  "identity": {
    "entraAgentId": "eab91a3d-3034-48d4-b540-65c56be4b9bd"
  },
  "runtime": {
    "type": "foundry",
    "endpoint": "https://aif-aiagents-vbds.services.ai.azure.com/api/projects/workshop-project",
   

In [1]:
import shutil

NVM_DIR_CANDIDATES = ["/usr/local/share/nvm", os.path.expanduser("~/.nvm")]
ATK_PACKAGE = "@microsoft/m365agentstoolkit-cli"

def ensure_node_and_npm() -> tuple[str, str]:
    """Return (node_path, npm_path), bootstrapping nvm + Node LTS if needed."""
    node = shutil.which("node")
    npm = shutil.which("npm")
    if node and npm:
        return node, npm

    nvm_dir = next((d for d in NVM_DIR_CANDIDATES if Path(d, "nvm.sh").exists()), None)
    if nvm_dir is None:
        raise RuntimeError(
            "Neither npm nor nvm is available. Please install Node.js LTS manually and re-run this cell."
        )

    print(f"Bootstrapping Node LTS via nvm at {nvm_dir} (one-time, ~30s)…")
    install_script = (
        f'export NVM_DIR="{nvm_dir}"; '
        'source "$NVM_DIR/nvm.sh"; '
        'nvm install --lts >/dev/null; '
        'nvm use --lts >/dev/null; '
        'echo "NODE=$(command -v node)"; '
        'echo "NPM=$(command -v npm)"'
    )
    out = subprocess.run(["bash", "-lc", install_script], capture_output=True, text=True, check=True)
    paths = dict(line.split("=", 1) for line in out.stdout.strip().splitlines() if "=" in line)
    node, npm = paths["NODE"], paths["NPM"]
    os.environ["PATH"] = f"{Path(npm).parent}:{os.environ['PATH']}"
    return node, npm

NODE_PATH, NPM_PATH = ensure_node_and_npm()
print("node:", NODE_PATH, "|", subprocess.check_output([NODE_PATH, "--version"], text=True).strip())
print("npm: ", NPM_PATH,  "|", subprocess.check_output([NPM_PATH, "--version"], text=True).strip())

print(f"\nInstalling {ATK_PACKAGE}@latest (1–2 min the first time)…")
subprocess.run([NPM_PATH, "install", "-g", f"{ATK_PACKAGE}@latest"], check=True)

ATK_PATH = shutil.which("atk") or str(Path(NPM_PATH).parent / "atk")
print("Agents Toolkit CLI:", ATK_PATH)

NameError: name 'os' is not defined

In [ ]:
import zipfile

# Minimal valid PNG bytes (1x1 transparent) so the manifest passes icon checks.
_PNG_1x1 = bytes.fromhex(
    "89504e470d0a1a0a0000000d49484452000000010000000108060000001f15c4"
    "890000000d49444154789c6300010000000500010d0a2db40000000049454e44"
    "ae426082"
)
(MANIFEST_DIR / "color.png").write_bytes(_PNG_1x1)
(MANIFEST_DIR / "outline.png").write_bytes(_PNG_1x1)

# Reference the icons from the manifest and rewrite it.
manifest["icons"] = {"color": "color.png", "outline": "outline.png"}
(MANIFEST_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))

PACKAGE_PATH = MANIFEST_DIR / "appPackage.zip"
with zipfile.ZipFile(PACKAGE_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    for name in ("manifest.json", "color.png", "outline.png"):
        z.write(MANIFEST_DIR / name, arcname=name)
print("App package built:", PACKAGE_PATH.resolve())
print("\nDownload this file to your local machine and install as described above.")

App package built: /workspaces/Microsoft-Foundry/src/samples/create-agent365-managed-agents/manifest/appPackage.zip

Now:
  • Option A: download this file and run the three `atk` commands on your laptop.
  • Option B: set up SSH port-forwarding, then run the next cell.
  • Option C: hand this file to your tenant admin and continue from Step 6.


In [ ]:
# Publishing via notebook is no longer supported. Use the manual route as described above.
print("Publishing via notebook skipped. Download the zip file and install locally with the atk CLI.")

Publish skipped (RUN_PUBLISH = False). See Option A in the markdown above.


## Publish and install the Agent 365 app package (manual)

Use the **Microsoft 365 Agents Toolkit CLI** (`atk`) to validate and install the manifest generated above into your tenant.

**Steps:**
1. Download the file `appPackage.zip` from the `manifest` folder to your local machine.
2. Open a terminal on your own laptop or use the Azure Cloud shell. 
3. Run the following commands:

```bash
atk validate --package-file appPackage.zip
atk install --file-path appPackage.zip --scope Shared
```

- You must be signed in to your Microsoft 365 tenant with `atk auth login m365`.
- Follow the CLI instructions to sign in.
- After installation, the agent will appear in the Microsoft 365 admin center under **Agents**.


Confirm the agent appears in the **Microsoft 365 admin center**:

1. Open <https://admin.microsoft.com> → **Agents**.
2. You should see **Foundry Lab Agent** with status *Published*.
3. Note the **Agent 365 manifest id** in the details panel — you will use it in the next step.

Set it here:

In [ ]:
def get_agent365_package_by_name(agent_name: str) -> dict:
    resp = graph_request(
        "GET",
        "/beta/copilot/admin/catalog/packages?$filter=supportedHosts/any(h:h eq 'Copilot')"
    )

    # Useful for debugging
    if resp.status_code >= 400:
        print(resp.status_code)
        print(resp.text)

    resp.raise_for_status()

    agents = resp.json().get("value", [])

    for agent in agents:
        display_name = agent.get("displayName", "").strip().lower()
        name = agent.get("name", "").strip().lower()

        if display_name == agent_name.strip().lower() or name == agent_name.strip().lower():
            print("Found agent:")
            print(f"Package ID:  {agent.get('id')}")
            print(f"Manifest ID: {agent.get('manifestId')}")
            return agent

    available = [a.get("displayName") or a.get("name") for a in agents]
    raise ValueError(
        f"No agent found with name: {agent_name}. "
        f"Available agents: {available}"
    )

# Use this function only if you want to look up the manifest id via Graph, otherwise you can fill it in manually.
# agent = get_agent365_package_by_name("Foundry Lab Agent")

# AGENT365_PACKAGE_ID = agent["id"]
# AGENT365_MANIFEST_ID = agent.get("manifestId")

Graph request failed
Status: 403
URL: https://graph.microsoft.com/beta/copilot/admin/catalog/packages?$filter=supportedHosts/any(h:h%20eq%20'Copilot')
Body: {"error":{"code":"UnknownError","message":"","innerError":{"date":"2026-05-11T11:52:25","request-id":"7d515e83-6e66-4ff0-8a58-0139a3b0253d","client-request-id":"7d515e83-6e66-4ff0-8a58-0139a3b0253d"}}}
403
{"error":{"code":"UnknownError","message":"","innerError":{"date":"2026-05-11T11:52:25","request-id":"7d515e83-6e66-4ff0-8a58-0139a3b0253d","client-request-id":"7d515e83-6e66-4ff0-8a58-0139a3b0253d"}}}


HTTPStatusError: Client error '403 Forbidden' for url 'https://graph.microsoft.com/beta/copilot/admin/catalog/packages?$filter=supportedHosts/any(h:h%20eq%20'Copilot')'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403

In [ ]:
AGENT365_MANIFEST_ID = "foundry-lab-agent"  # Fill in manually based on the admin center; override if your tenant rewrites it.

## Step 6 &mdash; Verify the agent is fully in sync

We now query each plane and assert the identifiers line up. A green ✓ table
means the agent is registered in **Entra**, running in **Foundry**, and
governed by **Agent 365** &mdash; with consistent identifiers in all three.

In [ ]:
# Entra plane
entra_resp = graph_request("GET", f"/applications/{ENTRA_AGENT_OBJECT_ID}?$select=id,appId,displayName")
entra_resp.raise_for_status()
entra = entra_resp.json()

# Foundry plane (via the SDK — addresses the agent by name, not by id).
foundry_agent = project_client.agents.get(agent_name=FOUNDRY_AGENT_NAME)
foundry_versions = list(project_client.agents.list_versions(agent_name=FOUNDRY_AGENT_NAME))
foundry_latest = max(foundry_versions, key=lambda v: int(v.version))
foundry_binding = (foundry_latest.metadata or {}).get("entraAgentId")

# Agent 365 plane (Graph)
a365_resp = graph_request("GET", f"/copilot/agents/{AGENT365_MANIFEST_ID}")
a365_resp.raise_for_status()
a365 = a365_resp.json()

rows = [
    ("Entra appId",                  entra["appId"],                                   ENTRA_AGENT_ID),
    ("Foundry agent name",           foundry_agent.name,                               FOUNDRY_AGENT_NAME),
    ("Foundry latest version",       str(foundry_latest.version),                     FOUNDRY_AGENT_VERSION),
    ("Foundry → Entra binding",      foundry_binding,                                  ENTRA_AGENT_ID),
    ("Agent 365 → Entra binding",    a365.get("identity", {}).get("entraAgentId"),     ENTRA_AGENT_ID),
    ("Agent 365 → Foundry binding",  a365.get("runtime", {}).get("agentName"),         FOUNDRY_AGENT_NAME),
]
print(f"{'Check':<32} {'Status':<6} {'Value'}")
all_ok = True
for name, actual, expected in rows:
    ok = actual == expected
    all_ok &= ok
    print(f"{name:<32} {'✓' if ok else '✗':<6} {actual}")
assert all_ok, "Sync check failed — see rows above."
print("\nAgent is fully in sync across Entra, Foundry, and Agent 365.")

HTTPStatusError: Client error '400 Bad Request' for url 'https://graph.microsoft.com/v1.0/copilot/agents/foundry-lab-agent'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/400

## Step 7 &mdash; Apply and test Agent 365 policies

Five policy categories. For each one: a short *why*, the *configuration*
(portal or Graph), and an executable *test* cell that produces a
deterministic pass/fail.

### 7a. DLP &mdash; block sensitive data leaving the agent

**Configure (portal):**

1. Go to <https://purview.microsoft.com> → **Data Loss Prevention** → **Policies** → **Create policy**.
2. Template: *Custom*. Location: enable **AI agents** and target *Foundry Lab Agent*.
3. Rule: *Content contains* sensitive info type **Credit Card Number**, count ≥ 1.
4. Action: **Block** the response and notify the user.
5. Turn the policy **On**.

**Test:** ask the agent to summarise a ticket whose body contains a fake card
number. The agent should refuse or return a redacted response.

In [ ]:
fake_ticket = {
    "ticket_id": "T-1042",
    # Synthetic test card; matches Luhn so DLP detectors trigger.
    "body": "Customer reports failed payment on card 4539 1488 0343 6467 — please retry.",
}

resp = agent_app.run_turn(
    openai_client,
    MODEL_DEPLOYMENT_NAME,
    handle,
    f"Summarise this ticket as JSON: {json.dumps(fake_ticket)}",
)
text = agent_app.final_text(resp)
lower = text.lower()
blocked = (
    any(token in lower for token in ["blocked", "redacted", "cannot share", "policy"])
    or "4539" not in text
)
print("Agent response:\n", text)
print("\nDLP test:", "PASS — card not present in response" if blocked else "FAIL — card leaked")

### 7b. Conditional Access &mdash; restrict where the agent can run

**Configure (portal):**

1. Entra → **Protection** → **Conditional Access** → **New policy**.
2. Assignments → **Workload identities** → select the Entra Agent ID `foundry-lab-agent`.
3. Conditions → **Locations** → exclude *Trusted named locations* (or set whatever boundary your tenant uses).
4. Grant → **Block access**.
5. Enable the policy.

**Test:** request a token *as the agent identity* from a non-trusted network.
Expect `AADSTS53003` (blocked by CA).

In [ ]:
# Manual / out-of-band test:
#   az login --service-principal -u <ENTRA_AGENT_ID> -p <secret> --tenant <tenant>
#   az account get-access-token --resource https://ai.azure.com
# When run from a blocked location, the call returns AADSTS53003.
print("Run the two `az` commands above from a non-trusted network.")
print("Expected error code: AADSTS53003 — Access has been blocked by Conditional Access policies.")

### 7c. Tool allow-list &mdash; restrict which actions the agent may invoke

**Configure (Graph):** patch the Agent 365 record so only `lookup_policy` is
allowed. `summarize_ticket` calls will then be refused at the governance plane.

In [ ]:
patch = graph_request(
    "PATCH",
    f"/copilot/agents/{AGENT365_MANIFEST_ID}",
    json={"governance": {"allowedActions": ["lookup_policy"]}},
)
patch.raise_for_status()
print("Allow-list applied.")

# Test: ask something that needs summarize_ticket.
# `tool_filter` simulates the Agent 365 governance refusal locally so the
# test is deterministic even before the policy propagates to the runtime.
refused = {"hit": False}
def allowlist(name: str) -> bool:
    if name != "lookup_policy":
        refused["hit"] = True
        return False
    return True

resp = agent_app.run_turn(
    openai_client,
    MODEL_DEPLOYMENT_NAME,
    handle,
    "Summarise ticket T-9 with body 'printer is offline since Monday morning'.",
    tool_filter=allowlist,
)
print("Agent response:\n", agent_app.final_text(resp))
print("\nAllow-list test:", "PASS" if refused["hit"] else "FAIL — summarize_ticket was not blocked")

### 7d. Audit &mdash; confirm the agent's actions are observable

Query the **Microsoft Graph audit logs** for events emitted by our test runs.
Look for `AgentRunStarted` and `AgentToolInvoked` events attributed to the
Entra Agent ID.

In [ ]:
# Audit ingestion can lag a few minutes — give it a moment.
time.sleep(60)

filt = (
    f"initiatedBy/app/appId eq '{ENTRA_AGENT_ID}'"
    " and (activityDisplayName eq 'AgentRunStarted'"
    " or activityDisplayName eq 'AgentToolInvoked')"
)
audit = graph_request(
    "GET",
    f"/auditLogs/agentEvents?$filter={httpx.QueryParams({'f': filt})['f']}&$top=10",
)
audit.raise_for_status()
events = audit.json().get("value", [])
for e in events:
    print(e.get("activityDateTime"), e.get("activityDisplayName"), e.get("id"))
print("\nAudit test:", "PASS — events found" if events else "FAIL — no events (wait longer or verify diagnostic settings)")

### 7e. Lifecycle &mdash; suspend, restore, retire

Lifecycle is the most operational of the five &mdash; admins must be able to
*pause* an agent that misbehaves without uninstalling it.

In [ ]:
def set_agent_state(state: str) -> dict:
    r = graph_request("PATCH", f"/copilot/agents/{AGENT365_MANIFEST_ID}", json={"state": state})
    r.raise_for_status()
    return r.json()


print("Suspending…", set_agent_state("suspended")["state"])

# Calling a suspended agent should now fail at the runtime boundary.
try:
    agent_app.run_turn(openai_client, MODEL_DEPLOYMENT_NAME, handle, "ping")
    print("Lifecycle test: FAIL — call succeeded while suspended")
except Exception as exc:  # noqa: BLE001
    print("Lifecycle test: PASS — runtime rejected with:", type(exc).__name__)

print("Restoring…", set_agent_state("published")["state"])

## Step 8 &mdash; Cleanup (optional)

Run the next cell to remove everything this lab created. Leave it commented out
if you want to keep exploring.

In [ ]:
# Uncomment to fully tear down everything this lab created.
#
# # Remove the sideloaded app package from the tenant catalog.
# !{ATK_PATH} uninstall --mode manifest-id --manifest-id {manifest["id"]}
#
# # Delete every version of the Foundry agent, then the agent itself.
# for v in list(project_client.agents.list_versions(agent_name=FOUNDRY_AGENT_NAME)):
#     project_client.agents.delete_version(agent_name=FOUNDRY_AGENT_NAME, agent_version=v.version)
# project_client.agents.delete(agent_name=FOUNDRY_AGENT_NAME)
#
# # Delete the Entra Agent ID app object.
# graph_request("DELETE", f"/applications/{ENTRA_AGENT_OBJECT_ID}")
# print("Cleanup complete.")

## Recap

You built a pro-code Foundry agent, gave it an **Entra Agent ID**, bound the
Foundry runtime to that identity, generated and published an **Agent 365
manifest**, verified the agent is **fully in sync** across all three planes,
and applied + tested the five governance categories that matter in production:
**DLP, Conditional Access, tool allow-list, audit, and lifecycle**.

Next steps:

* Replace the mock tools in [`agent_app.py`](agent_app.py) with real business APIs.
* Add Microsoft Graph permissions to the Entra Agent ID and call M365 services as the agent.
* Wire up an outcome ledger ([`../create-outcome-aware-agents`](../create-outcome-aware-agents/README.md)) to attribute business value to each governed run.